In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
df_devices = pd.read_csv('../data/tablas-actualizadas/devices.csv')
df_notifications = pd.read_csv('../data/tablas-actualizadas/notifications.csv')
df_transactions = pd.read_csv('../data/tablas-actualizadas/transactions.csv')
df_users = pd.read_csv('../data/tablas-actualizadas/users.csv')

display(df_devices.head(5), df_notifications.head(5), df_transactions.head(5), df_users.head(5))

In [ ]:
## cambiar el nombre a la tabla

df_devices = df_devices.rename(columns={"string_field_0": "brand_device", "string_field_1": "user_id"})
df_notifications = df_notifications.rename(columns={'created_date': 'create_date_notification'})
df_transactions = df_transactions.rename(columns={'created_date': 'create_date_transaction'})
df_users = df_users.rename(columns={'created_date': 'create_date_user'})

display(df_devices, df_notifications, df_transactions, df_users)

In [ ]:
for df in [df_users, df_devices, df_notifications, df_transactions]:
    if 'Unnamed: 0' in df.columns:
        df.drop(columns='Unnamed: 0', inplace=True)

display(df_devices, df_notifications, df_transactions, df_users)

In [ ]:
# Empieza con df_users
df = df_users.copy()

# Merge con devices
df = df.merge(df_devices, on='user_id', how='left')
df['has_device'] = df['brand_device'].notna()

# Merge con notifications
df = df.merge(df_notifications, on='user_id', how='left')
df['has_notification'] = df['create_date_notification'].notna()

# Merge con transactions
df = df.merge(df_transactions, on='user_id', how='left')
df['has_transaction'] = df['transaction_id'].notna()


In [ ]:
df

In [ ]:
df.columns

In [ ]:
from google.cloud import bigquery

client = bigquery.Client()
query = """
    SELECT * FROM `numeric-advice-452700-j9.neo_bank_.perfil_usuario`
"""
df_perfil_usuario = client.query(query).to_dataframe()

df_perfil_usuario

# 3. 👥 Perfiles de usuario y uso del producto

plan

In [ ]:
# Asegúrate de que df_perfil_usuario esté cargado

# 1) Eliminar duplicados por user_id para no contar al mismo usuario varias veces
df_unicos = df_perfil_usuario.drop_duplicates(subset='user_id')

# 2) Filtrar usuarios con plan definido (no nulo ni cadena vacía)
df_planes = df_unicos[df_unicos['plan'].notna() & (df_unicos['plan'] != '')]

# 3) Contar usuarios únicos por tipo de plan
distribucion_planes = df_planes['plan'].value_counts().reset_index()
distribucion_planes.columns = ['plan', 'usuarios']

# 4) Ordenar para que se vea más claro
distribucion_planes = distribucion_planes.sort_values(by='usuarios', ascending=False).reset_index(drop=True)

print(distribucion_planes)


In [ ]:
import pandas as pd
import plotly.express as px

# 1) Eliminar duplicados por user_id para no contar al mismo usuario varias veces
df_unicos = df_perfil_usuario.drop_duplicates(subset='user_id')

# 2) Filtrar usuarios con plan definido (no nulo ni vacío)
df_planes = df_unicos[df_unicos['plan'].notna() & (df_unicos['plan'] != '')]

# 3) Contar usuarios únicos por tipo de plan
distribucion_planes = df_planes['plan'].value_counts().reset_index()
distribucion_planes.columns = ['plan', 'usuarios']

# 4) Ordenar para que se vea más claro
distribucion_planes = distribucion_planes.sort_values(by='usuarios', ascending=False).reset_index(drop=True)

# Mostrar tabla
display(distribucion_planes)

# 5) Gráfica de barras
fig_plan = px.bar(
    distribucion_planes,
    x='plan',
    y='usuarios',
    color='plan',
    text='usuarios',
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={'plan': 'Plan', 'usuarios': 'Usuarios únicos'}
)

fig_plan.update_traces(textposition='outside')
fig_plan.update_layout(
    title='Distribución de usuarios por plan',
    xaxis_title='Tipo de plan',
    yaxis_title='Número de usuarios'
)

fig_plan.show()


edad

In [ ]:
# 1) Filtrar usuarios con grupo de edad no nulo
df_edad = df_perfil_usuario[df_perfil_usuario['age_group'].notna()]

# 2) Agrupar por grupo de edad y contar usuarios únicos
edad_grouped = (
    df_edad.groupby('age_group')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
    .sort_values(by='usuarios', ascending=False)
    .reset_index(drop=True)
)

# Mostrar tabla
display(edad_grouped, )

# 3) Gráfica de barras
import plotly.express as px

fig_edad = px.bar(
    edad_grouped,
    x='age_group',
    y='usuarios',
    color='age_group',
    text='usuarios',
    labels={'age_group': 'Grupo de edad', 'usuarios': 'Usuarios únicos'},
    color_discrete_sequence=px.colors.sequential.Sunset
)

fig_edad.update_traces(textposition='outside')
fig_edad.update_layout(
    title='Distribución de usuarios por grupo de edad',
    xaxis_title='Grupo de edad',
    yaxis_title='Número de usuarios'
)

fig_edad.show()


__Por país__

In [ ]:
# 1) Filtrar registros válidos con lat/lon y country_name no nulos
df_map = df_perfil_usuario[
    df_perfil_usuario['country_name'].notna() &
    df_perfil_usuario['lat'].notna() &
    df_perfil_usuario['lon'].notna()
]

# 2) Agrupar por país y coordenadas, contando usuarios únicos
usuarios_pais_mapa = (
    df_map.groupby(['country_name', 'lat', 'lon'])['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
    .sort_values('usuarios', ascending=False)
)

# Mostrar tabla
display(usuarios_pais_mapa, usuarios_pais_mapa['usuarios'].sum())

# 3) Crear mapa interactivo con scatter_geo
fig_mapa = px.scatter_geo(
    usuarios_pais_mapa,
    lat='lat',
    lon='lon',
    size='usuarios',
    hover_name='country_name',
    size_max=40,
    projection='natural earth',
    color='usuarios',
    color_continuous_scale='Plasma',
    labels={'usuarios': 'Usuarios'},
    title='Distribución geográfica de usuarios por país'
)

fig_mapa.update_geos(
    showcountries=True,
    showcoastlines=True,
    showland=True,
    landcolor="lightgray"
)

fig_mapa.update_layout(
    margin=dict(l=0, r=0, t=50, b=0),
    coloraxis_colorbar=dict(title="Usuarios")
)

fig_mapa.show()



__por ciudad__

In [ ]:
# Filtrar registros válidos
df_ciudades = df_perfil_usuario[
    df_perfil_usuario['city'].notna() & df_perfil_usuario['country_name'].notna()
]

# Agrupar por país y ciudad, contar usuarios únicos
usuarios_ciudades = (
    df_ciudades.groupby(['country_name', 'city'])['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
    .sort_values('usuarios', ascending=False)
)

# Seleccionar el top 20 por número de usuarios
top_ciudades = usuarios_ciudades.head(20)

# Mostrar tabla
display(top_ciudades)

# Graficar
fig_top_ciudades = px.bar(
    top_ciudades,
    x='usuarios',
    y='city',
    color='usuarios',
    text='usuarios',
    orientation='h',
    labels={'city': 'Ciudad', 'usuarios': 'Usuarios'},
    title='Top 20 ciudades con más usuarios (por país)',
    facet_col='country_name',  # opcional: dividir por país si quieres
)

fig_top_ciudades.update_layout(height=800, showlegend=False)
fig_top_ciudades.show()


# Transacciones por grupo de edad

In [ ]:
# Agrupar por usuario para obtener una fila única por user_id
group_usuarios = df_perfil_usuario.groupby('user_id').agg({
    'age_group': 'first',
    'num_transactions': 'first'  # Esto es el total de transacciones por usuario
}).reset_index()

# Luego agrupar por grupo de edad sumando el total de transacciones
transacciones_por_edad = group_usuarios.groupby('age_group')['num_transactions'].sum().reset_index()

# Ordenar por número de transacciones para mayor claridad
transacciones_por_edad = transacciones_por_edad.sort_values(by='num_transactions', ascending=False)

# Mostrar tabla
display(transacciones_por_edad, transacciones_por_edad['num_transactions'].sum())

# Crear gráfico de barras
import plotly.express as px

fig_txn_edad = px.bar(
    transacciones_por_edad,
    x='age_group',
    y='num_transactions',
    color='num_transactions',
    labels={'age_group': 'Grupo de edad', 'num_transactions': 'Total de transacciones'},
    color_continuous_scale='viridis',
    text='num_transactions',
    title='Total de transacciones por grupo de edad'
)

fig_txn_edad.update_traces(textposition='outside')
fig_txn_edad.update_layout(
    xaxis_title='Grupo de edad',
    yaxis_title='Total de transacciones'
)

fig_txn_edad.show()


# % de conversión por tipo de plan


In [ ]:
# Asegurarnos de trabajar con usuarios únicos
usuarios_unicos = df_perfil_usuario.drop_duplicates(subset='user_id')

# Agrupar por plan y calcular métricas
conversion_por_plan = usuarios_unicos.groupby('plan')['converted'].agg(['count', 'sum']).reset_index()
conversion_por_plan['conversion_rate'] = round((conversion_por_plan['sum'] / conversion_por_plan['count']) * 100, 2)

# Ordenar por % de conversión descendente
conversion_por_plan = conversion_por_plan.sort_values(by='conversion_rate', ascending=False)

# Mostrar tabla
display(conversion_por_plan)

fig_conversion = px.bar(
    conversion_por_plan,
    x='plan',
    y='conversion_rate',
    labels={'conversion_rate': '% de conversión', 'plan': 'Tipo de plan'},
    text='conversion_rate',
    color='conversion_rate',
    color_continuous_scale='gnbu',
    title='% de conversión por tipo de plan'
)

fig_conversion.update_traces(textposition='outside')
fig_conversion.update_layout(
    xaxis_title='Tipo de plan',
    yaxis_title='% de conversión'
)

fig_conversion.show()


In [ ]:
# Asegurarte de que user_settings_crypto_unlocked sea numérico (si viene como objeto o str)
df_perfil_usuario['user_settings_crypto_unlocked'] = pd.to_numeric(
    df_perfil_usuario['user_settings_crypto_unlocked'], errors='coerce'
)

# Reemplazar NaN con 0 (considerar que si no está habilitado es como no activado)
df_perfil_usuario['crypto_flag'] = df_perfil_usuario['user_settings_crypto_unlocked'].fillna(0).apply(lambda x: 'Sí' if x == 1 else 'No')

# Agrupar y contar usuarios únicos
uso_crypto = (
    df_perfil_usuario.groupby('crypto_flag')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
    .sort_values(by='usuarios', ascending=False)
)

# Mostrar tabla
display(uso_crypto)

# Crear gráfico de pastel
import plotly.express as px

fig_crypto = px.pie(
    uso_crypto,
    values='usuarios',
    names='crypto_flag',
    color_discrete_sequence=px.colors.sequential.RdBu,
    title='Funcionalidad cripto activada'
)
fig_crypto.show()


In [ ]:
# Filtrar solo las ciudades con más usuarios
top_ciudades = usuarios_por_ciudad.head(20)
top_ciudades

In [ ]:
# visualización
fig = px.bar(
    top_ciudades,
    x='usuarios',
    y='city',
    orientation='h',
    color='usuarios',
    color_continuous_scale='Tealgrn',
    title='Top 20 ciudades con más usuarios únicos',
    labels={'city': 'Ciudad', 'usuarios': 'Usuarios únicos'}
)
fig.show()

__Distribución por canal__

In [ ]:
# Agrupar por canal y contar usuarios únicos
usuarios_por_canal = (
    df.groupby('channel')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
    .sort_values('usuarios', ascending=False)
)
usuarios_por_canal

In [ ]:
# Visualizar
fig = px.bar(
    usuarios_por_canal,
    x='channel',
    y='usuarios',
    color='usuarios',
    title='Distribución de usuarios por canal',
    labels={'channel': 'Canal de adquisición', 'usuarios': 'Usuarios únicos'},
    color_continuous_scale='burg'
)
fig.show()

__por device__

In [ ]:
#Agrupar por dispositivo y contar usuarios únicos
usuarios_por_dispositivo = (
    df.groupby('brand_device')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
    .sort_values(by='usuarios', ascending=False)
)
usuarios_por_dispositivo

In [ ]:
# visualización
fig = px.bar(
    usuarios_por_dispositivo,
    x='brand_device',
    y='usuarios',
    title='Distribución de usuarios por tipo de dispositivo',
    labels={'usuarios': 'Usuarios únicos', 'brand_device': 'Dispositivo'},
    color='usuarios',
    color_continuous_scale='cividis'
)
fig.show()



# Número de transacciones por segmento

In [ ]:
# Agrupar por usuario para obtener una fila por user_id con su edad y total de transacciones
usuarios_unicos = df.groupby('user_id').agg({
    'age_group': 'first',
    'num_transactions': 'first'  # o 'sum', dependiendo cómo esté el dato original
}).reset_index()

# Ahora agrupar por grupo de edad
transacciones_por_edad = usuarios_unicos.groupby('age_group')['num_transactions'].sum().reset_index()

# nostrar
df_transactions = pd.read_csv('../data/transactions.csv')
df_txn = df_transactions['transaction_id'].nunique()
display(usuarios_unicos, transacciones_por_edad, transacciones_por_edad['num_transactions'].sum(), df_txn)

In [ ]:
# Crear gráfico de barras
fig = px.bar(
    transacciones_por_edad,
    x='age_group',
    y='num_transactions',
    title='Número total de transacciones por grupo de edad',
    labels={'age_group': 'Grupo de edad', 'num_transactions': 'Total de transacciones'},
    color='num_transactions',
    color_continuous_scale='viridis'
)
fig.show()

# % de conversión por grupo

In [ ]:
usuarios_unicos = df.drop_duplicates(subset='user_id')
usuarios_unicos

In [ ]:
# Agrupar por plan y calcular métricas de conversión
conversion_por_plan = usuarios_unicos.groupby('plan')['converted'].agg(['count', 'sum']).reset_index()
conversion_por_plan['conversion_rate'] = (conversion_por_plan['sum'] / conversion_por_plan['count']) * 100
conversion_por_plan

In [ ]:
# visualizar
fig = px.bar(
    conversion_por_plan,
    x='plan',
    y='conversion_rate',
    title='% de conversión por tipo de plan',
    labels={'conversion_rate': '% de conversión', 'plan': 'Tipo de plan'},
    text='conversion_rate',
    color='conversion_rate',
    color_continuous_scale='gnbu'
)
fig.show()

METAL tiene una tasa de conversión de ~1.18%, lo que significa que 1.18 de cada 100 usuarios con plan METAL convirtieron.

STANDARD tiene una tasa de conversión más baja: ~0.66%.

PREMIUM_FREE y PREMIUM_OFFER están en 0%, posiblemente porque nadie con esos planes convirtió o hay muy pocos usuarios.

In [ ]:
df.columns

# Uso de funcionalidades crypto

In [ ]:
# Agrupar por uso de cripto y contar usuarios únicos
uso_crypto = df.groupby('user_settings_crypto_unlocked')['user_id'].nunique().reset_index()
uso_crypto.columns = ['Crypto habilitado', 'Usuarios únicos']

uso_crypto

In [ ]:
# Convertir valores a etiquetas legibles
uso_crypto['Crypto habilitado'] = uso_crypto['Crypto habilitado'].map({
    'True': 'Sí',
    'False': 'No',
    True: 'Sí',
    False: 'No'
})

uso_crypto


In [ ]:
# Gráfico de pastel
fig = px.pie(
    uso_crypto,
    values='Usuarios únicos',
    names='Crypto habilitado',
    title='Distribución de usuarios con funcionalidad cripto activada',
    color_discrete_sequence=px.colors.sequential.RdBu
)

fig.show()

In [ ]:
print(df[['country', 'plan', 'age']].info())
print(df[['country', 'plan']].dropna().nunique())

In [ ]:
print(repr(df['country'].unique()))  # <- esto revela si hay espacios ocultos

In [ ]:
# Supón que eliges uno de los países válidos
pais_seleccionado = 'FR'  # o cualquier otro que veas en df['country'].unique()

# Supón que eliges algunos planes
planes_seleccionados = ['STANDARD', 'PREMIUM', 'METAL']  # ajusta según tu dataset

# Rango de edad (simulado)
edad_min = int(df['age'].min())
edad_max = int(df['age'].max())
rango_edad = (edad_min, edad_max)  # o (24, 65), etc.

print("Total:", len(df))

df1 = df[df['country'] == pais_seleccionado]
print("Filtrado por país:", len(df1))

df2 = df1[df1['plan'].isin(planes_seleccionados)]
print("Filtrado por plan:", len(df2))

df3 = df2[df2['age'].between(rango_edad[0], rango_edad[1])]
print("Filtrado por edad:", len(df3))

df_filtrado = df3
df_filtrado.head()  # Mostrar muestra


# 4. ⚠️ Identificación de churn y recomendaciones

__churn__

In [ ]:
# Paso 1: Última fecha del dataset
fecha_max = df['create_date_transaction'].max()

fecha_max

In [ ]:

# Paso 2: Fecha de última transacción por usuario
ultima_txn = df.groupby('user_id')['create_date_transaction'].max().reset_index()
ultima_txn.columns = ['user_id', 'ultima_txn']
ultima_txn


In [ ]:

# Paso 3: Calcular días de inactividad
ultima_txn['dias_inactivo'] = (fecha_max - ultima_txn['ultima_txn']).dt.days
ultima_txn

In [ ]:
# Paso 4: Marcar usuarios como churned si llevan más de 60 días sin transaccionar
ultima_txn['churned'] = ultima_txn['dias_inactivo'] > 90
ultima_txn

In [ ]:
# Paso 5: Churn rate
churn_rate = ultima_txn['churned'].mean() * 100
churn_rate

# calcular churn en df

In [ ]:
# Paso 1: Obtener la última fecha de transacción del dataset
fecha_corte = df['create_date_transaction'].max()
fecha_corte

In [ ]:

# Paso 2: Convertir fechas a naive (sin timezone), por si acaso
fecha_corte = fecha_corte.tz_localize(None) if fecha_corte.tzinfo else fecha_corte
df['create_date_transaction'] = df['create_date_transaction'].dt.tz_localize(None)

In [ ]:

# Paso 3: Calcular días desde última transacción
df['dias_inactivo'] = (fecha_corte - df['create_date_transaction']).dt.days
df

In [ ]:

# Paso 4: Marcar como churned si lleva más de 60 días sin transaccionar
df['churned'] = df['dias_inactivo'] > 60

In [ ]:
df[(df['has_transaction'] == True) & (df['churned'] == True)]

In [ ]:

# df.to_csv('../data/df_n.csv')

In [ ]:
# Filtrar usuarios notificados (por push o email) pero que NO se convirtieron
notificados_no_convertidos = df[
    ((df['attributes_notifications_marketing_push'] == 1) | 
     (df['attributes_notifications_marketing_email'] == 1)) & 
    (df['converted'] == 0)
]
notificados_no_convertidos


In [ ]:

# Crear nueva columna con canal de notificación
def canal(row):
    if row['attributes_notifications_marketing_push'] == 1 and row['attributes_notifications_marketing_email'] == 1:
        return 'Ambos'
    elif row['attributes_notifications_marketing_push'] == 1:
        return 'Push'
    elif row['attributes_notifications_marketing_email'] == 1:
        return 'Email'
    else:
        return 'Ninguno'

notificados_no_convertidos['canal_notificado'] = notificados_no_convertidos.apply(canal, axis=1)

notificados_no_convertidos

In [ ]:
# Agrupar por canal
canal_counts = notificados_no_convertidos['canal_notificado'].value_counts().reset_index()
canal_counts.columns = ['Canal', 'Usuarios no convertidos']

canal_counts

In [ ]:

# Graficar
fig = px.pie(
    canal_counts, 
    names='Canal', 
    values='Usuarios no convertidos', 
    title='Usuarios notificados pero no convertidos (por canal)'
)
fig.show()

In [ ]:
df

# Usuarios notificados pero no convertidos por tipo de plan

In [ ]:
# Filtrar usuarios notificados pero no convertidos
notificados_no_convertidos = df[(df['has_notification'] == True) & (df['converted'] == 0)]
notificados_no_convertidos

In [ ]:

# Agrupar por plan
usuarios_por_plan = notificados_no_convertidos.groupby('plan')['user_id'].nunique().reset_index()
usuarios_por_plan.columns = ['plan', 'usuarios']

usuarios_por_canal


In [ ]:

# Visualización
fig = px.bar(
    usuarios_por_plan,
    x='plan',
    y='usuarios',
    color='plan',
    title='Usuarios notificados pero no convertidos por tipo de plan',
    labels={'usuarios': 'Usuarios únicos', 'plan': 'Tipo de plan'}
)

fig.show()

# Usuarios notificados pero no convertidos por grupo de edad

In [ ]:
# Filtrar usuarios notificados pero no convertidos
notificados_no_convertidos = df[(df['has_notification'] == True) & (df['converted'] == 0)]
notificados_no_convertidos

In [ ]:

# Agrupar por grupo de edad
usuarios_por_edad = notificados_no_convertidos.groupby('age_group')['user_id'].nunique().reset_index()
usuarios_por_edad.columns = ['age_group', 'usuarios']
usuarios_por_edad

In [ ]:

# Ordenar grupos de edad si son strings
usuarios_por_edad = usuarios_por_edad.sort_values(by='age_group')
usuarios_por_edad

In [ ]:

# Visualizar
fig = px.bar(
    usuarios_por_edad,
    x='age_group',
    y='usuarios',
    color='age_group',
    title='Usuarios notificados pero no convertidos por grupo de edad',
    labels={'usuarios': 'Usuarios únicos', 'age_group': 'Grupo de edad'}
)

fig.show()

# Usuarios completamente inactivos (sin ninguna transacción) por grupo de edad

In [ ]:
import plotly.express as px

# Filtrar usuarios sin transacciones
usuarios_inactivos = df[df['has_transaction'] == False]
usuarios_inactivos


In [ ]:

# Agrupar por grupo de edad (puedes cambiarlo a 'plan', 'channel', etc.)
inactivos_por_edad = usuarios_inactivos.groupby('age_group')['user_id'].nunique().reset_index()
inactivos_por_edad.columns = ['age_group', 'usuarios']
inactivos_por_edad

In [ ]:

# Ordenar si es necesario
inactivos_por_edad = inactivos_por_edad.sort_values(by='age_group')
inactivos_por_edad

In [ ]:

# Visualizar
fig = px.bar(
    inactivos_por_edad,
    x='age_group',
    y='usuarios',
    color='age_group',
    title='Usuarios completamente inactivos (sin ninguna transacción) por grupo de edad',
    labels={'usuarios': 'Usuarios únicos', 'age_group': 'Grupo de edad'}
)

fig.show()

# inactivos (sin ninguna transacción) por plan

In [ ]:
usuarios_inactivos = df[df['has_transaction'] == False]
usuarios_inactivos

In [ ]:
inactivos_por_plan = usuarios_inactivos.groupby('plan')['user_id'].nunique().reset_index()
inactivos_por_plan.columns = ['plan', 'usuarios_inactivos']
inactivos_por_plan

In [ ]:
fig = px.bar(
    inactivos_por_plan.sort_values(by='usuarios_inactivos', ascending=False),
    x='plan',
    y='usuarios_inactivos',
    title='Usuarios inactivos por tipo de plan',
    labels={'usuarios_inactivos': 'Usuarios inactivos', 'plan': 'Tipo de plan'},
    color='usuarios_inactivos',
    color_continuous_scale='viridis'
)
fig.show()


# inactivos (que nunca hicieron transacción) agrupados por canal

In [ ]:
usuarios_inactivos = df[df['has_transaction'] == False]
usuarios_inactivos

In [ ]:
inactivos_por_canal = usuarios_inactivos.groupby('channel')['user_id'].nunique().reset_index()
inactivos_por_canal.columns = ['channel', 'usuarios_inactivos']
inactivos_por_canal

In [ ]:
fig = px.bar(
    inactivos_por_canal.sort_values(by='usuarios_inactivos', ascending=False),
    x='channel',
    y='usuarios_inactivos',
    title='Usuarios inactivos por canal de adquisición',
    labels={'usuarios_inactivos': 'Usuarios inactivos', 'channel': 'Canal'},
    color='usuarios_inactivos',
    color_continuous_scale='plasma'
)
fig.show()

# Días sin actividad desde alta

In [ ]:
usuarios_sin_txn = df[df['has_transaction'] == False].copy()
usuarios_sin_txn

In [ ]:
fecha_max = df['create_date_transaction'].max()
fecha_max


In [ ]:
# Obtener la zona horaria desde create_date_user
tz = usuarios_sin_txn['create_date_user'].dt.tz
tz


In [ ]:

# Asegurar que la fecha máxima tenga la misma zona horaria
fecha_max = pd.Timestamp(df['create_date_transaction'].max(), tz=tz)
fecha_max

In [ ]:

# Calcular días sin actividad
usuarios_sin_txn['dias_sin_actividad'] = (fecha_max - usuarios_sin_txn['create_date_user']).dt.days
usuarios_sin_txn

In [ ]:
bins = [0, 7, 14, 30, 60, 90, 180, 365, 9999]
labels = ['0-7 días', '8-14 días', '15-30 días', '31-60 días', '61-90 días', '91-180 días', '181-365 días', '365+ días']

usuarios_sin_txn['rango_dias_sin_actividad'] = pd.cut(usuarios_sin_txn['dias_sin_actividad'], bins=bins, labels=labels)
dias_inactivos = usuarios_sin_txn.groupby('rango_dias_sin_actividad')['user_id'].nunique().reset_index()
dias_inactivos.columns = ['rango_dias', 'usuarios']


In [ ]:
fig = px.bar(
    dias_inactivos,
    x='rango_dias',
    y='usuarios',
    title='Usuarios sin actividad por rango de días desde su alta',
    labels={'rango_dias': 'Días sin actividad', 'usuarios': 'Usuarios únicos'},
    color='usuarios',
    color_continuous_scale='magma'
)
fig.show()


In [ ]:
df.groupby('churned')['user_id'].nunique()

In [ ]:
df

In [ ]:
df_unicos = df.drop_duplicates(subset='user_id', keep='first')
df_con_transaccion = df_unicos[df_unicos['has_transaction'] == True]
df_con_transaccion.groupby('churned')['user_id'].nunique()

# dashboard v2.0

In [ ]:
# Asume que df es tu DataFrame con usuarios únicos y las columnas calculadas

# Métrica 1: Usuarios únicos
total_users = df['user_id'].nunique()
total_users

In [ ]:

# Métrica 2: % de conversión (usuarios que convirtieron al menos una vez)
conversion_rate = (df['converted'].sum() / total_users) * 100
conversion_rate

In [ ]:

# Métrica 3: % de churn (usuarios activos que luego abandonaron)
# Asegúrate de usar un df de usuarios únicos
df_unicos = df.drop_duplicates(subset='user_id')

total_users = df_unicos['user_id'].nunique()
churned_users = df_unicos[(df_unicos['churned'] == True) & (df_unicos['has_transaction'] == True)]['user_id'].nunique()

churn_rate = (churned_users / total_users) * 100

display(df_unicos, total_users, churned_users, churn_rate)


In [ ]:
# Métrica 4: Promedio de días a la primera transacción (entre usuarios que transaccionaron)

# 1) Obtener la primera transacción de cada usuario
df_txn = df.dropna(subset=['create_date_transaction'])
df_first_txn = df_txn.groupby('user_id')['create_date_transaction'].min().reset_index()
df_first_txn.columns = ['user_id', 'first_transaction_date']

# 2) Traer fecha de creación de usuario
df_user_creation = df[['user_id', 'create_date_user']].drop_duplicates()

# 3) Merge para calcular días a la primera transacción
df_dias_txn = pd.merge(df_user_creation, df_first_txn, on='user_id', how='left')

# --- ⚠️ Corregir zona horaria para evitar errores ---
df_dias_txn['first_transaction_date'] = df_dias_txn['first_transaction_date'].dt.tz_localize(None)
df_dias_txn['create_date_user'] = df_dias_txn['create_date_user'].dt.tz_localize(None)

# Calcular días a la primera transacción
df_dias_txn['dias_a_primera_txn'] = (
    df_dias_txn['first_transaction_date'] - df_dias_txn['create_date_user']
).dt.days

# 4) Merge con el df principal para incorporar dias_a_primera_txn
df_copy = df.drop_duplicates(subset='user_id').merge(
    df_dias_txn[['user_id', 'dias_a_primera_txn']],
    on='user_id',
    how='left'
)

avg_days_to_first_txn = df_copy['dias_a_primera_txn'].dropna().mean()

avg_days_to_first_txn


In [ ]:
df

# Evolución semanal de usuarios nuevos vs. activos

In [ ]:
# 1) Asegúrate de que las fechas de transacción están en datetime
df_n_i = df.copy()
df_n_i['create_date_transaction'] = pd.to_datetime(df_n_i['create_date_transaction'], errors='coerce')



In [ ]:

# 2) Crea la columna de semana de la transacción (usando inicio de semana: lunes)
df_n_i['semana_transaccion'] = df_n_i['create_date_transaction'].dt.to_period('W').apply(
    lambda r: r.start_time if pd.notnull(r) else pd.NaT
)


In [ ]:

# 3) Usuarios activos por semana (usuarios únicos que transaccionaron)
usuarios_activos = (
    df_n_i.dropna(subset=['semana_transaccion'])  # Asegura eliminar NaT aquí
      .groupby('semana_transaccion')['user_id']
      .nunique()
      .reset_index(name='usuarios_activos')
)
usuarios_activos

In [ ]:

# 4) Usuarios nuevos por semana (basado en fecha de creación de usuario)
df_n_i['create_date_user'] = pd.to_datetime(df_n_i['create_date_user'], errors='coerce')
df_n_i['semana_creacion'] = df_n_i['create_date_user'].dt.to_period('W').apply(
    lambda r: r.start_time if pd.notnull(r) else pd.NaT
)
usuarios_nuevos = (
    df_n_i.dropna(subset=['semana_creacion'])  # Asegura eliminar NaT aquí también
      .groupby('semana_creacion')['user_id']
      .nunique()
      .reset_index(name='usuarios_nuevos')
)

usuarios_nuevos

In [ ]:

# 5) Fusiona ambos resultados
df_evolucion = pd.merge(
    usuarios_nuevos,
    usuarios_activos,
    left_on='semana_creacion',
    right_on='semana_transaccion',
    how='outer'
)

df_evolucion

In [ ]:

# 6) Unifica la columna de semana
df_evolucion['semana'] = df_evolucion['semana_creacion'].combine_first(df_evolucion['semana_transaccion'])


In [ ]:

# 7) Elimina semanas NaT o inválidas
df_evolucion = df_evolucion.dropna(subset=['semana']).sort_values('semana')


In [ ]:

# 8) Graficar con Plotly Express
fig = px.line(
    df_evolucion,
    x='semana',
    y=['usuarios_nuevos', 'usuarios_activos'],
    labels={'value': 'Cantidad de usuarios', 'variable': 'Tipo de usuario', 'semana': 'Semana'},
    title='Evolución semanal de usuarios nuevos vs. activos'
)
fig.update_layout(xaxis=dict(tickformat='%Y-%m-%d'))  # Formato de fechas en eje X
fig.show()

In [ ]:
print("Última fecha de creación de usuario:", df['create_date_user'].max())
print("Última fecha de transacción:", df['create_date_transaction'].max())

In [ ]:
df_evolucion

In [ ]:
df_evolucion['usuarios_nuevos'].sum(), df_evolucion['usuarios_activos'].sum()

In [ ]:
df_evolucion['usuarios_nuevos'].sum(), df_evolucion['usuarios_activos'].sum()

In [ ]:
df

In [ ]:
# 1) Cohorte: semana de registro
df['cohort_week'] = df['create_date_user'].dt.to_period('W').apply(lambda r: r.start_time)
df


In [ ]:

# 2) Semana de actividad (semana de transacción)
df['activity_week'] = df['create_date_transaction'].dt.to_period('W').apply(lambda r: r.start_time if pd.notnull(r) else pd.NaT)
df

In [ ]:

# 3) Filtrar solo usuarios con transacción
df_active = df.dropna(subset=['activity_week'])
df_active

In [ ]:

# 4) Calcular semanas desde el registro correctamente
df_active['weeks_since_signup'] = (
    (df_active['activity_week'] - df_active['cohort_week']).dt.days // 7
)
df_active

In [ ]:

# 5) Usuarios únicos por cohorte y semana desde registro
retention = (
    df_active.groupby(['cohort_week', 'weeks_since_signup'])['user_id']
    .nunique()
    .reset_index(name='active_users')
)
retention

In [ ]:

# 6) Número de usuarios en cada cohorte (tamaño base)
cohort_sizes = (
    df.groupby('cohort_week')['user_id']
    .nunique()
    .reset_index(name='cohort_size')
)
cohort_sizes

In [ ]:

# 7) Merge para calcular % de retención
retention = retention.merge(cohort_sizes, on='cohort_week')
retention['retention_rate'] = retention['active_users'] / retention['cohort_size']

retention

In [ ]:

# 8) Pivotear para matriz de retención
retention_matrix = retention.pivot(index='cohort_week', columns='weeks_since_signup', values='retention_rate')
retention_matrix

In [ ]:

fig = px.imshow(
    retention_matrix,
    labels=dict(x='Semanas desde registro', y='Cohorte de registro', color='Tasa de retención'),
    color_continuous_scale='Blues',
    title='Curva de retención por cohortes semanales'
)
fig.show()

# Primera transacción

In [ ]:
# 1) Calcular días hasta primera transacción
df_merged = df.dropna(subset=['create_date_transaction']).copy()
df_merged['create_date_transaction'] = df_merged['create_date_transaction'].dt.tz_localize(None)
df_merged['create_date_user'] = df_merged['create_date_user'].dt.tz_localize(None)
df_merged['dias_a_primera_txn'] = (df_merged['create_date_transaction'] - df_merged['create_date_user']).dt.days

df_merged

In [ ]:

# Ahora calcula los días a la primera transacción
df_merged['dias_a_primera_txn'] = (
    df_merged['create_date_transaction'] - df_merged['create_date_user']
).dt.days

df_merged

In [ ]:

# 2) Quedarse solo con el primer registro de cada usuario para no duplicar conversiones
df_first_txn = df_merged.sort_values('dias_a_primera_txn').drop_duplicates(subset='user_id', keep='first')
df_first_txn

In [ ]:

# 3) Crear columnas booleanas: si convirtió en 1/7/30 días
df_first_txn['converted_1d'] = df_first_txn['dias_a_primera_txn'] <= 1
df_first_txn['converted_7d'] = df_first_txn['dias_a_primera_txn'] <= 7
df_first_txn['converted_30d'] = df_first_txn['dias_a_primera_txn'] <= 30
df_first_txn['converted_60d'] = df_first_txn['dias_a_primera_txn'] <= 60
df_first_txn['converted_90d'] = df_first_txn['dias_a_primera_txn'] <= 90
df_first_txn['converted_120d'] = df_first_txn['dias_a_primera_txn'] <= 120
df_first_txn['converted_mayor_120d'] = df_first_txn['dias_a_primera_txn'] > 120
df_first_txn

In [ ]:

# 4) Calcular el total de usuarios registrados
total_users = df['user_id'].nunique()
total_users

In [ ]:

# 5) Calcular tasas de conversión correctamente
conversion_1d = df_first_txn['converted_1d'].sum() / total_users * 100
conversion_7d = df_first_txn['converted_7d'].sum() / total_users * 100
conversion_30d = df_first_txn['converted_30d'].sum() / total_users * 100
conversion_60d = df_first_txn['converted_60d'].sum() / total_users * 100
conversion_90d = df_first_txn['converted_90d'].sum() / total_users * 100
conversion_120d = df_first_txn['converted_120d'].sum() / total_users * 100
converted_mayor_120d = df_first_txn['converted_mayor_120d'].sum() / total_users * 100

conversion_1d, conversion_7d, conversion_30d, conversion_60d, conversion_90d, conversion_120d, converted_mayor_120d

In [ ]:

conversion_data = {
    'Periodo': ['1 día', '7 días', '30 días', '60 días', '90 días', '120 días', 'Mayor a 120 días'],
    'Conversión (%)': [conversion_1d, conversion_7d, conversion_30d,  conversion_60d,  conversion_90d, conversion_120d, converted_mayor_120d]
}

conversion_data

In [ ]:

fig = px.bar(
    conversion_data,
    x='Periodo',
    y='Conversión (%)',
    text='Conversión (%)',
    title='% de usuarios que convierten en 1, 7 y 30 días',
    color='Periodo',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 100])
fig.show()


Cerca del 35% de los usuarios convierten dentro de los primeros 120 días, mientras que un 65% nunca realiza una transacción o la hace mucho más tarde, lo que sugiere oportunidades en onboarding y activación temprana.

In [ ]:
# Filtrar usuarios con al menos una transacción
df_txn = df.dropna(subset=['create_date_transaction'])

# Obtener primera transacción por usuario
df_first_txn = (
    df_txn.groupby('user_id')['create_date_transaction']
    .min()
    .reset_index()
    .rename(columns={'create_date_transaction': 'first_transaction_date'})
)

df_txn, df_first_txn

In [ ]:
df_user_creation = df[['user_id', 'create_date_user']].drop_duplicates()
df_user_creation

In [ ]:
df_merged = df_user_creation.merge(
    df_first_txn,
    on='user_id',
    how='inner'  # Solo usuarios que tienen al menos una transacción
)
df_merged

In [ ]:
df_merged['create_date_user'] = df_merged['create_date_user'].dt.tz_localize(None)
df_merged['first_transaction_date'] = df_merged['first_transaction_date'].dt.tz_localize(None)


In [ ]:
df_merged['dias_a_primera_txn'] = (
    df_merged['first_transaction_date'] - df_merged['create_date_user']
).dt.days


In [ ]:
df_merged

In [ ]:
print(df_merged['dias_a_primera_txn'].describe())


In [ ]:
df_merged['user_id'].nunique()

In [ ]:
fig = px.histogram(
    df_merged,
    x='dias_a_primera_txn',
    nbins=300,
    title='Distribución del tiempo hasta la primera transacción',
    labels={'dias_a_primera_txn': 'Días hasta primera transacción'},
    color_discrete_sequence=['#00BFC4']
)

fig.update_layout(
    xaxis_title='Días hasta la primera transacción',
    yaxis_title='Número de usuarios',
    bargap=0.1
)

fig.show()

# Usuarios notificados pero no convertidos 

In [ ]:
df.columns

In [ ]:
# Filtrar usuarios notificados pero sin transacción
usuarios_no_convertidos = df[
    (df['has_notification'] == True) & (df['has_transaction'] == False)
].copy()
usuarios_no_convertidos

# notificados no convertidos

In [ ]:
df.groupby('has_notification')['user_id'].nunique()

In [ ]:
# Agrupar por canal y contar usuarios únicos
no_conv_por_canal = (
    usuarios_no_convertidos.groupby('channel')['user_id']
    .nunique()
    .reset_index(name='usuarios_no_convertidos')
    .sort_values('usuarios_no_convertidos', ascending=False)
)
no_conv_por_canal['usuarios_no_convertidos'].sum()

In [ ]:

# Graficar
fig = px.bar(
    no_conv_por_canal,
    x='channel',
    y='usuarios_no_convertidos',
    text='usuarios_no_convertidos',
    title='Usuarios notificados pero no convertidos por canal',
    labels={'channel': 'Canal', 'usuarios_no_convertidos': 'Usuarios sin conversión'}
)
fig.update_traces(textposition='outside')
fig.show()


In [ ]:
no_conv_por_plan = (
    usuarios_no_convertidos.groupby('plan')['user_id']
    .nunique()
    .reset_index(name='usuarios_no_convertidos')
    .sort_values('usuarios_no_convertidos', ascending=False)
)

fig = px.bar(
    no_conv_por_plan,
    x='plan',
    y='usuarios_no_convertidos',
    text='usuarios_no_convertidos',
    title='Usuarios notificados pero no convertidos por plan',
    labels={'plan': 'Plan', 'usuarios_no_convertidos': 'Usuarios sin conversión'}
)
fig.update_traces(textposition='outside')
fig.show()


In [ ]:
no_conv_por_edad = (
    usuarios_no_convertidos.groupby('age_group')['user_id']
    .nunique()
    .reset_index(name='usuarios_no_convertidos')
    .sort_values('age_group')
)

fig = px.bar(
    no_conv_por_edad,
    x='age_group',
    y='usuarios_no_convertidos',
    text='usuarios_no_convertidos',
    title='Usuarios notificados pero no convertidos por grupo de edad',
    labels={'age_group': 'Grupo de edad', 'usuarios_no_convertidos': 'Usuarios sin conversión'}
)
fig.update_traces(textposition='outside')
fig.show()


In [ ]:
# Agrupar por canal y estado de conversión
df_grouped = (
    df[df['has_notification']]
    .groupby(['channel', 'converted'])['user_id']
    .nunique()
    .reset_index()
    .pivot(index='channel', columns='converted', values='user_id')
    .fillna(0)
    .reset_index()
)
df_grouped


In [ ]:

df_grouped.columns = ['Canal', 'No_convirtieron', 'Convirtieron']  # Reordenar columnas
df_grouped

In [ ]:

fig = px.bar(
    df_grouped,
    x='Canal',
    y=['No_convirtieron', 'Convirtieron'],
    labels={'value': 'Usuarios', 'variable': 'Estado'},
    title='Usuarios notificados: Convirtieron vs. No Convirtieron por canal',
    barmode='stack',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.show()

In [ ]:
# Filtrar usuarios activos e inactivos
usuarios_activos = df[df['has_transaction'] == True]
usuarios_inactivos = df[df['has_transaction'] == False]

# Agrupar activos por grupo de edad
activos_por_edad = (
    usuarios_activos.groupby('age_group')['user_id']
    .nunique()
    .reset_index(name='usuarios_activos')
)

# Agrupar inactivos por grupo de edad
inactivos_por_edad = (
    usuarios_inactivos.groupby('age_group')['user_id']
    .nunique()
    .reset_index(name='usuarios_inactivos')
)

# Unir ambos dataframes
comparacion_edad = pd.merge(
    activos_por_edad, inactivos_por_edad,
    on='age_group', how='outer'
).fillna(0).sort_values('age_group')

# Graficar apilada
import plotly.express as px

fig_comparacion_edad = px.bar(
    comparacion_edad,
    x='age_group',
    y=['usuarios_activos', 'usuarios_inactivos'],
    barmode='stack',  # <-- ¡aquí el cambio importante!
    labels={
        'value': 'Usuarios',
        'variable': 'Estado',
        'age_group': 'Grupo de edad'
    },
    title='Usuarios activos vs. inactivos por grupo de edad (Apilado)',
    color_discrete_sequence=px.colors.qualitative.Set2_r,
    text_auto=True
)

fig_comparacion_edad.update_layout(yaxis_title='Número de usuarios')
fig_comparacion_edad.show()


In [ ]:
usuarios_activos['user_id'].nunique(), usuarios_inactivos['user_id'].nunique()

In [ ]:
comparacion_edad['usuarios_inactivos'].sum()

In [ ]:
# Agrupar usuarios únicos por estado de churn
churn_summary = (
    df.drop_duplicates(subset='user_id')
    .groupby('churned')['user_id']
    .nunique()
    .reset_index(name='usuarios')
)
churn_summary


In [ ]:

# Convertir True/False a etiquetas más amigables
churn_summary['Estado'] = churn_summary['churned'].map({True: 'Churned', False: 'No Churned'})
churn_summary

In [ ]:

# Crear gráfico de pastel
fig_churn = px.pie(
    churn_summary,
    names='Estado',
    values='usuarios',
    title='Distribución de usuarios churned vs. no churned',
    color='Estado',
    color_discrete_map={'Churned': '#FF6F61', 'No Churned': '#6BA292'},  # Colores personalizados
    hole=0.4  # Para hacer un donut chart
)

fig_churn.update_traces(textinfo='percent+label')

fig_churn.show()

In [ ]:
# Agrupar usuarios únicos por plan y estado de churn
churn_by_plan = (
    df.drop_duplicates(subset='user_id')
    .groupby(['plan', 'churned'])['user_id']
    .nunique()
    .reset_index(name='usuarios')
)
churn_by_plan


In [ ]:

# Convertir churn True/False a etiquetas legibles
churn_by_plan['Estado'] = churn_by_plan['churned'].map({True: 'Churned', False: 'No Churned'})
churn_by_plan['usuarios'].sum()

In [ ]:

# Graficar barras apiladas por plan
fig_churn_plan = px.bar(
    churn_by_plan,
    x='plan',
    y='usuarios',
    color='Estado',
    barmode='stack',
    title='Distribución de churned vs. no churned por plan',
    color_discrete_map={'Churned': '#FF6F61', 'No Churned': '#6BA292'},
    text_auto=True
)

fig_churn_plan.update_layout(yaxis_title='Número de usuarios')
fig_churn_plan.show()

In [ ]:
df_ = pd.read_csv("https://drive.google.com/uc?export=download&id=13b2-OSvLH_wMbgUtBD41b-JaLVsrJ3Sh")
print(df_.columns)


In [ ]:
# import gdown
# import os

# # Descarga solo si el archivo aún no existe
# if not os.path.exists("../data/df.csv"):
#     url = "https://drive.google.com/uc?id=13b2-OSvLH_wMbgUtBD41b-JaLVsrJ3Sh"
#     gdown.download(url, "../data/df.csv", quiet=False)

# df = pd.read_csv("../data/df.csv", parse_dates=["create_date_user", "create_date_transaction"])


In [ ]:
df.columns

In [ ]:
df[df['user_id'] == 'user_1303']

In [ ]:

df_transactions[df_transactions['user_id']=='user_1303']